# CMFQuery REST API Tester

Exercises all major endpoints documented at
[CMFQuery REST API](https://hewlettpackard.github.io/cmf/api/public/cmfquery_rest_api/)
against a live cmf-server.

All endpoints live under `BASE_URL = <server>/api/v1` and return a common `APIResponse`
envelope: `{status, code, data, message, errors, meta}`. The helper below unwraps `data`
and prints one status line per call.

The notebook is self-seeding: it discovers a real pipeline, then real execution IDs, stage
names/IDs, execution UUIDs, and artifact names/IDs from the server, and feeds them into the
later endpoints — no hard-coded IDs.

In [1]:
import json
import requests
import pandas as pd
from urllib.parse import quote

BASE_URL = "http://ai02.labs.hpecorp.net/api/v1"   # cmf-server (nginx proxies /api -> server)
session = requests.Session()

def call(method, path, *, params=None, body=None):
    """Call an endpoint, print a status line, and return the APIResponse 'data' field."""
    r = session.request(method, f"{BASE_URL}{path}", params=params, json=body, timeout=60)
    try:
        payload = r.json()
    except ValueError:
        print(f"{method:4} {path} -> HTTP {r.status_code} (non-JSON response)")
        return None
    print(f"{method:4} {path} -> HTTP {r.status_code} [{payload.get('status')}] {payload.get('message')}")
    if payload.get("errors"):
        print("     errors:", payload["errors"])
    return payload.get("data")

def enc(segment):
    """Percent-encode a path segment (artifact/stage names contain '/' and ':')."""
    return quote(str(segment), safe="")

def records(data):
    """Extract the list of records from a data payload (dict-of-list or plain list)."""
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        for v in data.values():
            if isinstance(v, list):
                return v
    return []

def scalar(data):
    """Extract a single scalar from a data payload (dict with one scalar value, or scalar)."""
    if isinstance(data, dict):
        for v in data.values():
            if isinstance(v, (int, float, str)):
                return v
    return data

def preview(data, n=5):
    """Preview record-style data as a small DataFrame."""
    rows = records(data)
    if rows and isinstance(rows[0], dict):
        return pd.json_normalize(rows).head(n)
    return data

pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 200)
print("Base URL:", BASE_URL)

Base URL: http://ai02.labs.hpecorp.net/api/v1


## 1. Pipelines

| Endpoint | Purpose |
|---|---|
| `GET /pipelines/names` | all pipeline names |
| `GET /pipelines/{pipeline_name}/id` | numeric pipeline id |
| `GET /pipelines/{pipeline_name}/json` | full metadata dump (optional `?exec_uuid=`) |
| `GET /pipelines/sync/{last_sync_time}/json` | metadata modified after a sync timestamp |

In [2]:
# GET /pipelines/names
data = call("GET", "/pipelines/names")
PIPELINE = records(data)[0]
print("Using pipeline:", PIPELINE)

GET  /pipelines/names -> HTTP 200 [success] Pipeline names retrieved successfully
Using pipeline: Test-env


In [3]:
# GET /pipelines/{pipeline_name}/id
data = call("GET", f"/pipelines/{enc(PIPELINE)}/id")
PIPELINE_ID = int(data["pipeline_id"] if isinstance(data, dict) and "pipeline_id" in data else scalar(data))
print("Pipeline id:", PIPELINE_ID)

GET  /pipelines/Test-env/id -> HTTP 200 [success] Pipeline ID retrieved successfully
Pipeline id: 1


In [4]:
# GET /pipelines/{pipeline_name}/json  — full mlmd dump for one pipeline
data = call("GET", f"/pipelines/{enc(PIPELINE)}/json")
dump = json.loads(data) if isinstance(data, str) else data
if isinstance(dump, dict) and "json_payload" in dump:
    dump = json.loads(dump["json_payload"]) if isinstance(dump["json_payload"], str) else dump["json_payload"]
pipeline_block = dump["Pipeline"][0] if isinstance(dump, dict) and "Pipeline" in dump else dump
stage_names = [s.get("name") for s in pipeline_block.get("stages", [])]
print("Stages in dump:", stage_names)

GET  /pipelines/Test-env/json -> HTTP 200 [success] Pipeline JSON retrieved successfully
Stages in dump: ['Test-env/Evaluate', 'Test-env/Train', 'Test-env/Featurize', 'Test-env/Prepare']


In [5]:
# Same endpoint filtered to one execution: GET /pipelines/{p}/json?exec_uuid=...
first_stage = pipeline_block["stages"][0]
first_exec_uuid = first_stage["executions"][0]["properties"]["Execution_uuid"].split(",")[0]
data = call("GET", f"/pipelines/{enc(PIPELINE)}/json", params={"exec_uuid": first_exec_uuid})
print("Filtered dump size (chars):", len(json.dumps(data)))

GET  /pipelines/Test-env/json -> HTTP 200 [success] Pipeline JSON retrieved successfully
Filtered dump size (chars): 4517


In [6]:
# GET /pipelines/sync/{last_sync_time}/json  — everything modified after epoch-ms 0
data = call("GET", "/pipelines/sync/0/json")
print("Sync payload preview:", json.dumps(data)[:200], "...")

GET  /pipelines/sync/0/json -> HTTP 200 [success] Pipelines JSON extracted successfully
Sync payload preview: {"Pipeline": [{"stages": [{"executions": [{"type": "Test-env/Evaluate", "name": "", "events": [{"type": 3, "artifact": {"type": "Environment", "create_time_since_epoch": 1793706684493, "custom_propert ...


## 2. Executions

| Endpoint | Purpose |
|---|---|
| `GET /executions/pipeline/{pipeline_name}` | executions of a pipeline |
| `GET /executions/stages/name/{stage_name}` | execution details for a stage name |
| `GET /executions/stages/name/{stage_name}/list` | execution objects for a stage name |
| `GET /executions/stages/id/{stage_id}` | executions for a stage id (optional `?execution_uuid=`) |
| `POST /executions/batch-get` | details for a list of execution ids |
| `POST /executions/batch-summary` | summary for a list of execution ids |
| `GET /executions/id/{id}/artifacts` | artifacts tied to one execution |
| `GET /executions/id/{id}/parents/ids` | one-hop parent execution ids |
| `POST /executions/parents/batch-get` | one-hop parents for several executions |
| `POST /executions/ancestors/batch-get` | full upstream ancestry |
| `POST /executions/artifacts/batch-get` | artifacts for several executions |

In [7]:
# GET /executions/pipeline/{pipeline_name}
data = call("GET", f"/executions/pipeline/{enc(PIPELINE)}")
exec_df = pd.json_normalize(records(data))
EXEC_IDS = [int(i) for i in exec_df["id"]]
STAGE_NAME = exec_df["Context_Type"].iloc[0]
STAGE_ID = int(exec_df["Context_ID"].iloc[0])
EXEC_UUID = str(exec_df["Execution_uuid"].iloc[0]).split(",")[0]
LAST_EXEC_ID = EXEC_IDS[0] if len(EXEC_IDS) == 1 else int(exec_df.sort_values("id")["id"].iloc[-1])
print(f"{len(EXEC_IDS)} executions | stage {STAGE_NAME!r} (id {STAGE_ID}) | uuid {EXEC_UUID}")
exec_df[["id", "Context_ID", "Context_Type", "Execution_uuid"]]

GET  /executions/pipeline/Test-env -> HTTP 200 [success] Pipeline executions retrieved successfully
8 executions | stage 'Test-env/Evaluate' (id 5) | uuid effa2112-d77b-11f1-8e32-1b603906cbd8


,id,Context_ID,Context_Type,Execution_uuid
0,8,5,Test-env/Evaluate,effa2112-d77b-11f1-8e32-1b603906cbd8
1,7,5,Test-env/Evaluate,271bb782-d77c-11f1-9d1e-ed9d59c2225e
2,6,4,Test-env/Train,eb451de8-d77b-11f1-9f7c-7728aabab821
3,5,4,Test-env/Train,22707b00-d77c-11f1-821c-ddf24d2c56f8
4,4,3,Test-env/Featurize,c6c41672-d77b-11f1-a61c-17e03a460857
5,3,3,Test-env/Featurize,07f27896-d77c-11f1-b0ab-0b6555ed3885
6,2,2,Test-env/Prepare,c158872c-d77b-11f1-a66a-797ec8d5a5a2
7,1,2,Test-env/Prepare,037848ea-d77c-11f1-bff7-87eaa9b7b0a6


In [8]:
# GET /executions/stages/name/{stage_name}  and  .../list
by_stage = call("GET", f"/executions/stages/name/{enc(STAGE_NAME)}")
by_stage_list = call("GET", f"/executions/stages/name/{enc(STAGE_NAME)}/list")
preview(by_stage)

GET  /executions/stages/name/Test-env%2FEvaluate -> HTTP 200 [success] Executions associated with pipeline stage retrieved successfully
GET  /executions/stages/name/Test-env%2FEvaluate/list -> HTTP 200 [success] Executions associated with pipeline stage retrieved successfully


,Context_ID,Context_Type,Execution,Execution_type_name,Execution_uuid,Git_End_Commit,...,Pipeline_Type,Pipeline_id,custom_properties_Python_Env,custom_properties_original_create_time_since_epoch,id,name
0,5,Test-env/Evaluate,"['src/test.py', 'artifacts/model', 'artifacts/...",Test-env/Evaluate,effa2112-d77b-11f1-8e32-1b603906cbd8,,...,Test-env,1,cmf_artifacts/python_env_496a933a0d7c1de9824a2...,1793706273227,8,
1,5,Test-env/Evaluate,"['src/test.py', 'artifacts/model', 'artifacts/...",Test-env/Evaluate,271bb782-d77c-11f1-9d1e-ed9d59c2225e,,...,Test-env,1,cmf_artifacts/python_env_496a933a0d7c1de9824a2...,1793706365720,7,


In [9]:
# GET /executions/stages/id/{stage_id}  — plain, then filtered by execution_uuid
by_id = call("GET", f"/executions/stages/id/{STAGE_ID}")
by_id_filtered = call("GET", f"/executions/stages/id/{STAGE_ID}", params={"execution_uuid": EXEC_UUID})
preview(by_id_filtered)

GET  /executions/stages/id/5 -> HTTP 200 [success] Executions for stage retrieved successfully
GET  /executions/stages/id/5 -> HTTP 200 [success] Executions for stage retrieved successfully


,id,type_id,name,external_id,create_time_since_epoch,last_update_time_since_epoch,...,properties.Execution_type_name,properties.Context_ID,properties.Execution_uuid,properties.Git_End_Commit,custom_properties.Python_Env,custom_properties.original_create_time_since_epoch
0,8,20,,,1793706684604,1793706684606,...,Test-env/Evaluate,5,effa2112-d77b-11f1-8e32-1b603906cbd8,,cmf_artifacts/python_env_496a933a0d7c1de9824a2...,1793706273227


In [10]:
# POST /executions/batch-get  and  /executions/batch-summary
batch = call("POST", "/executions/batch-get", body={"exe_ids": EXEC_IDS[:3]})
summary = call("POST", "/executions/batch-summary", body={"exe_ids": EXEC_IDS[:3]})
preview(batch)

POST /executions/batch-get -> HTTP 200 [success] Executions retrieved successfully
POST /executions/batch-summary -> HTTP 200 [success] Execution summary retrieved successfully


{'exe_ids': [8, 7, 6],
 'executions': [{'Context_ID': 4,
   'Context_Type': 'Test-env/Train',
   'Execution': "['src/train.py', 'artifacts/features', 'artifacts/model']",
   'Execution_type_name': 'Test-env/Train',
   'Execution_uuid': 'eb451de8-d77b-11f1-9f7c-7728aabab821',
   'Git_End_Commit': '',
   'Git_Repo': 'https://github.com/GA-FDP/mode_classifier.git',
   'Git_Start_Commit': 'a2373fdb5be274b3536dde89024ba5055fc8d669',
   'Pipeline_Type': 'Test-env',
   'Pipeline_id': 1,
   'custom_properties_Python_Env': 'cmf_artifacts/python_env_496a933a0d7c1de9824a23070a91e008.txt',
   'custom_properties_min_split': 64.0,
   'custom_properties_n_est': 100.0,
   'custom_properties_original_create_time_since_epoch': '1793706265324',
   'custom_properties_seed': 20170428.0},
  {'Context_ID': 5,
   'Context_Type': 'Test-env/Evaluate',
   'Execution': "['src/test.py', 'artifacts/model', 'artifacts/features', 'artifacts/test_results']",
   'Execution_type_name': 'Test-env/Evaluate',
   'Execution

In [11]:
# GET /executions/id/{execution_id}/artifacts
exec_artifacts = call("GET", f"/executions/id/{LAST_EXEC_ID}/artifacts")
# GET /executions/id/{execution_id}/parents/ids   (pipeline_id optional)
parent_ids = call("GET", f"/executions/id/{LAST_EXEC_ID}/parents/ids", params={"pipeline_id": PIPELINE_ID})
print("Parent execution ids of", LAST_EXEC_ID, "->", parent_ids)
preview(exec_artifacts)

GET  /executions/id/8/artifacts -> HTTP 200 [success] Artifacts for execution retrieved successfully
GET  /executions/id/8/parents/ids -> HTTP 200 [success] One-hop parent execution ids retrieved successfully
Parent execution ids of 8 -> {'execution_id': 8, 'pipeline_id': 1, 'parent_execution_ids': [3, 4, 5, 6], 'total_parent_execution_ids': 4}


,Commit,create_time_since_epoch,custom_properties_avg_prec,custom_properties_original_create_time_since_epoch,custom_properties_roc_auc,event,...,model_name,model_type,name,type,uri,url
0,0e3c310f31eff210851daf160ad993ae,1793706684493,NaN,NaN,NaN,INPUT,...,NaN,NaN,cmf_artifacts/python_env_496a933a0d7c1de9824a2...,Environment,0e3c310f31eff210851daf160ad993ae,Test-env:/lustre/royann/cmf/files/md5/0e/3c310...
1,db38e93d1ec3b65588a17f2d60e2928e,1793706684573,NaN,1793706271062,NaN,INPUT,...,RandomForestClassifier:default,RandomForestClassifier,artifacts/model/model.pkl:db38e93d1ec3b65588a1...,Model,db38e93d1ec3b65588a17f2d60e2928e,Test-env:/lustre/royann/cmf/files/md5/db/38e93...
2,a4babb0b21f092267774e849ace94f83,1793706684541,NaN,1793706262773,NaN,INPUT,...,NaN,NaN,artifacts/features/test.pkl:a4babb0b21f0922677...,Dataset,a4babb0b21f092267774e849ace94f83,Test-env:/lustre/royann/cmf/files/md5/a4/babb0...
3,NaN,1793706684615,0.572594,1793706275834,0.952194,OUTPUT,...,NaN,NaN,metrics:f18760ee-d77b-11f1-8e32-1b603906cbd8:8,Metrics,f18760ee-d77b-11f1-8e32-1b603906cbd8,NaN


In [12]:
# POST /executions/parents/batch-get, /executions/ancestors/batch-get, /executions/artifacts/batch-get
parents_batch = call("POST", "/executions/parents/batch-get",
                     body={"execution_id": EXEC_IDS[:3], "pipeline_id": PIPELINE_ID})
# NOTE: ancestors/batch-get currently always returns 404 "Parent executions not found" due to an
# upstream bug: cmfquery.get_all_parent_executions_by_id expects a nested list from
# get_one_hop_parent_executions, which actually returns a flat list, so nothing is ever collected.
ancestors_batch = call("POST", "/executions/ancestors/batch-get",
                       body={"execution_id": [LAST_EXEC_ID], "pipeline_id": PIPELINE_ID})
artifacts_batch = call("POST", "/executions/artifacts/batch-get", body={"exe_ids": EXEC_IDS[:3]})
preview(parents_batch)

POST /executions/parents/batch-get -> HTTP 200 [success] One-hop parent executions retrieved successfully
POST /executions/ancestors/batch-get -> HTTP 200 [error] Parent executions not found
     errors: [{'field': 'execution_id', 'message': 'Parent executions not found for execution ids [8]', 'code': None}]
POST /executions/artifacts/batch-get -> HTTP 200 [success] Artifacts for executions retrieved successfully


{'execution_id': [8, 7, 6],
 'pipeline_id': 1,
 'executions': [{'id': 4,
   'type_id': 16,
   'name': '',
   'external_id': '',
   'create_time_since_epoch': 1793706684544,
   'last_update_time_since_epoch': 1793706684546,
   'properties': {'Pipeline_id': 1,
    'Git_Repo': 'https://github.com/GA-FDP/mode_classifier.git',
    'Execution': "['src/featurize.py', 'artifacts/parsed', 'artifacts/features']",
    'Pipeline_Type': 'Test-env',
    'Git_Start_Commit': '38993e504576383d1abca3cf1eca1cde1d226934',
    'Context_Type': 'Test-env/Featurize',
    'Execution_type_name': 'Test-env/Featurize',
    'Context_ID': 3,
    'Execution_uuid': 'c6c41672-d77b-11f1-a61c-17e03a460857',
    'Git_End_Commit': ''},
   'custom_properties': {'max_features': 3000,
    'ngrams': 2,
    'Python_Env': 'cmf_artifacts/python_env_496a933a0d7c1de9824a23070a91e008.txt',
    'original_create_time_since_epoch': '1793706204080'}},
  {'id': 3,
   'type_id': 16,
   'name': '',
   'external_id': '',
   'create_time_si

## 3. Artifacts

| Endpoint | Purpose |
|---|---|
| `GET /artifacts` | all artifact names |
| `GET /artifacts/{pipeline_name}` | artifacts of a pipeline |
| `POST /artifacts/batch-get` | artifacts by id list |
| `GET /artifacts/name/{name}` | metadata by name |
| `GET /artifacts/name/{name}/dataframe` | dataframe form |
| `GET /artifacts/name/{name}/executions` | executions linked to artifact |
| `GET /artifacts/id/{id}/executions` | executions by artifact id |
| `GET /artifacts/name/{name}/parent-executions` | parent executions |
| `GET /artifacts/name/{name}/producer-execution` | producing execution |
| `GET /artifacts/name/{name}/children` | one-hop child artifacts |
| `GET /artifacts/name/{name}/descendants` | all downstream artifacts |
| `GET /artifacts/name/{name}/parents` | one-hop parent artifacts |
| `GET /artifacts/id/{id}/parents` | one-hop parents by id |
| `GET /artifacts/name/{name}/ancestors` | all upstream artifacts |
| `GET /artifacts/metrics/{metrics_name}` | metrics artifact metadata |

In [13]:
# GET /artifacts  — every artifact name on the server
all_names = records(call("GET", "/artifacts"))
print(f"{len(all_names)} artifacts total; first 3: {all_names[:3]}")

# GET /artifacts/{pipeline_name}
data = call("GET", f"/artifacts/{enc(PIPELINE)}")
art_df = pd.json_normalize(records(data))
ART_IDS = [int(i) for i in art_df["id"]]
art_df[["id", "name", "type", "uri"]]

GET  /artifacts -> HTTP 200 [success] Artifacts retrieved successfully
11 artifacts total; first 3: ['cmf_artifacts/python_env_496a933a0d7c1de9824a23070a91e008.txt:0e3c310f31eff210851daf160ad993ae', 'artifacts/data.xml.gz:236d9502e0283d91f689d7038b8508a2', 'artifacts/labels.csv:13e556fa36c8a2a1f711be954edaa805']
GET  /artifacts/Test-env -> HTTP 200 [success] Artifacts associated with pipeline retrieved successfully


,id,name,type,uri
0,11,metrics:f18760ee-d77b-11f1-8e32-1b603906cbd8:8,Metrics,f18760ee-d77b-11f1-8e32-1b603906cbd8
1,10,metrics:28af1a1c-d77c-11f1-9d1e-ed9d59c2225e:7,Metrics,28af1a1c-d77c-11f1-9d1e-ed9d59c2225e
2,8,cmf_artifacts/22707b00-d77c-11f1-821c-ddf24d2c...,Step_Metrics,12bd2c3831b14430234cf2bb74b71e19
3,9,artifacts/model/model.pkl:db38e93d1ec3b65588a1...,Model,db38e93d1ec3b65588a17f2d60e2928e
4,7,artifacts/features/test.pkl:a4babb0b21f0922677...,Dataset,a4babb0b21f092267774e849ace94f83
5,6,artifacts/features/train.pkl:dc07d9df71460e786...,Dataset,dc07d9df71460e7867a9c2e53ce9509d
6,1,cmf_artifacts/python_env_496a933a0d7c1de9824a2...,Environment,0e3c310f31eff210851daf160ad993ae
7,5,artifacts/parsed/test.tsv:6f597d341ceb7d8fbbe8...,Dataset,6f597d341ceb7d8fbbe88859a892ef81
8,4,artifacts/parsed/train.tsv:32b715ef0d71ff4c9e6...,Dataset,32b715ef0d71ff4c9e61f55b09c15e75
9,2,artifacts/data.xml.gz:236d9502e0283d91f689d703...,Dataset,236d9502e0283d91f689d7038b8508a2


In [14]:
# Pick anchors for the lineage endpoints:
#  - a Dataset artifact mid-lineage (has both parents and children)
#  - a Step_Metrics artifact for the metrics endpoint (get_metrics only searches type
#    'Step_Metrics', i.e. metrics logged via cmf.log_metric/commit_metrics — plain
#    'Metrics' artifacts are not found by it)
datasets = art_df[art_df["type"] == "Dataset"]["name"].tolist()
ART_NAME = next((n for n in datasets if "train" in n), datasets[0] if datasets else art_df["name"].iloc[0])
ART_ID = int(art_df.loc[art_df["name"] == ART_NAME, "id"].iloc[0])
metrics_rows = art_df[art_df["type"] == "Step_Metrics"]
METRICS_NAME = metrics_rows["name"].iloc[0] if not metrics_rows.empty else None
print(f"Anchor artifact: {ART_NAME!r} (id {ART_ID})")
print(f"Step metrics artifact: {METRICS_NAME!r}")

Anchor artifact: 'artifacts/features/train.pkl:dc07d9df71460e7867a9c2e53ce9509d' (id 6)
Step metrics artifact: 'cmf_artifacts/22707b00-d77c-11f1-821c-ddf24d2c56f8/metrics/training_metrics:12bd2c3831b14430234cf2bb74b71e19:7:2578d4e6-d77c-11f1-821c-ddf24d2c56f8'


In [15]:
# POST /artifacts/batch-get
batch = call("POST", "/artifacts/batch-get", body={"artifact_ids": ART_IDS[:5]})
preview(batch)

POST /artifacts/batch-get -> HTTP 200 [success] Artifacts retrieved successfully


{'artifact_ids': [11, 10, 8, 9, 7],
 'artifacts': [{'Commit': 'a4babb0b21f092267774e849ace94f83',
   'create_time_since_epoch': 1793706684541,
   'custom_properties_avg_prec': None,
   'custom_properties_original_create_time_since_epoch': '1793706262773',
   'custom_properties_roc_auc': None,
   'git_repo': 'https://github.com/GA-FDP/mode_classifier.git',
   'id': 7,
   'last_update_time_since_epoch': 1793706684541,
   'metrics_name': None,
   'model_framework': None,
   'model_name': None,
   'model_type': None,
   'name': 'artifacts/features/test.pkl:a4babb0b21f092267774e849ace94f83',
   'type': 'Dataset',
   'uri': 'a4babb0b21f092267774e849ace94f83',
   'url': 'Test-env:/lustre/royann/cmf/files/md5/a4/babb0b21f092267774e849ace94f83'},
  {'Commit': '12bd2c3831b14430234cf2bb74b71e19',
   'create_time_since_epoch': 1793706684570,
   'custom_properties_avg_prec': None,
   'custom_properties_original_create_time_since_epoch': '1793706362979',
   'custom_properties_roc_auc': None,
   'git

In [16]:
# GET /artifacts/name/{artifact_name}  and  .../dataframe
meta = call("GET", f"/artifacts/name/{enc(ART_NAME)}")
frame = call("GET", f"/artifacts/name/{enc(ART_NAME)}/dataframe")
preview(frame) if frame else meta

GET  /artifacts/name/artifacts%2Ffeatures%2Ftrain.pkl%3Adc07d9df71460e7867a9c2e53ce9509d -> HTTP 200 [success] Artifact retrieved successfully
GET  /artifacts/name/artifacts%2Ffeatures%2Ftrain.pkl%3Adc07d9df71460e7867a9c2e53ce9509d/dataframe -> HTTP 200 [success] Artifact dataframe retrieved successfully


,id,type,uri,name,create_time_since_epoch,last_update_time_since_epoch,git_repo,Commit,url,custom_properties_original_create_time_since_epoch
0,6,Dataset,dc07d9df71460e7867a9c2e53ce9509d,artifacts/features/train.pkl:dc07d9df71460e786...,1793706684539,1793706684539,https://github.com/GA-FDP/mode_classifier.git,dc07d9df71460e7867a9c2e53ce9509d,Test-env:/lustre/royann/cmf/files/md5/dc/07d9d...,1793706262156


In [17]:
# Executions linked to the artifact — by name, by id, parents, and producer
ex_by_name = call("GET", f"/artifacts/name/{enc(ART_NAME)}/executions")
ex_by_id = call("GET", f"/artifacts/id/{ART_ID}/executions")
parent_exec = call("GET", f"/artifacts/name/{enc(ART_NAME)}/parent-executions")
producer = call("GET", f"/artifacts/name/{enc(ART_NAME)}/producer-execution")
preview(producer) if records(producer) else producer

GET  /artifacts/name/artifacts%2Ffeatures%2Ftrain.pkl%3Adc07d9df71460e7867a9c2e53ce9509d/executions -> HTTP 200 [success] Executions for artifact retrieved successfully
GET  /artifacts/id/6/executions -> HTTP 200 [success] Executions for artifact retrieved successfully
GET  /artifacts/name/artifacts%2Ffeatures%2Ftrain.pkl%3Adc07d9df71460e7867a9c2e53ce9509d/parent-executions -> HTTP 200 [success] All parent executions retrieved successfully
GET  /artifacts/name/artifacts%2Ffeatures%2Ftrain.pkl%3Adc07d9df71460e7867a9c2e53ce9509d/producer-execution -> HTTP 200 [success] Producer execution retrieved successfully


{'artifact_name': 'artifacts/features/train.pkl:dc07d9df71460e7867a9c2e53ce9509d',
 'execution': {'id': 4,
  'type_id': 16,
  'name': '',
  'external_id': '',
  'create_time_since_epoch': 1793706684544,
  'last_update_time_since_epoch': 1793706684546,
  'properties': {'Pipeline_id': 1,
   'Git_End_Commit': '',
   'Git_Repo': 'https://github.com/GA-FDP/mode_classifier.git',
   'Pipeline_Type': 'Test-env',
   'Git_Start_Commit': '38993e504576383d1abca3cf1eca1cde1d226934',
   'Context_Type': 'Test-env/Featurize',
   'Execution_type_name': 'Test-env/Featurize',
   'Execution_uuid': 'c6c41672-d77b-11f1-a61c-17e03a460857',
   'Context_ID': 3,
   'Execution': "['src/featurize.py', 'artifacts/parsed', 'artifacts/features']"},
  'custom_properties': {'max_features': 3000,
   'ngrams': 2,
   'Python_Env': 'cmf_artifacts/python_env_496a933a0d7c1de9824a23070a91e008.txt',
   'original_create_time_since_epoch': '1793706204080'}}}

In [18]:
# Artifact-to-artifact lineage: children / descendants / parents (name + id) / ancestors
children = call("GET", f"/artifacts/name/{enc(ART_NAME)}/children", params={"pipeline_id": PIPELINE_ID})
descendants = call("GET", f"/artifacts/name/{enc(ART_NAME)}/descendants")
parents_by_name = call("GET", f"/artifacts/name/{enc(ART_NAME)}/parents")
parents_by_id = call("GET", f"/artifacts/id/{ART_ID}/parents")
ancestors = call("GET", f"/artifacts/name/{enc(ART_NAME)}/ancestors")
print("children:", len(records(children)), "| descendants:", len(records(descendants)),
      "| parents:", len(records(parents_by_name)), "| ancestors:", len(records(ancestors)))
preview(descendants)

GET  /artifacts/name/artifacts%2Ffeatures%2Ftrain.pkl%3Adc07d9df71460e7867a9c2e53ce9509d/children -> HTTP 200 [success] One-hop child artifacts retrieved successfully
GET  /artifacts/name/artifacts%2Ffeatures%2Ftrain.pkl%3Adc07d9df71460e7867a9c2e53ce9509d/descendants -> HTTP 200 [success] All child artifacts retrieved successfully
GET  /artifacts/name/artifacts%2Ffeatures%2Ftrain.pkl%3Adc07d9df71460e7867a9c2e53ce9509d/parents -> HTTP 200 [success] One-hop parent artifacts retrieved successfully
GET  /artifacts/id/6/parents -> HTTP 200 [success] One-hop parent artifacts retrieved successfully
GET  /artifacts/name/artifacts%2Ffeatures%2Ftrain.pkl%3Adc07d9df71460e7867a9c2e53ce9509d/ancestors -> HTTP 200 [success] All parent artifacts retrieved successfully
children: 2 | descendants: 4 | parents: 3 | ancestors: 5


,Commit,create_time_since_epoch,custom_properties_avg_prec,custom_properties_original_create_time_since_epoch,custom_properties_roc_auc,id,...,model_name,model_type,name,type,uri,url
0,12bd2c3831b14430234cf2bb74b71e19,1793706684570,NaN,1793706362979,NaN,8,...,NaN,NaN,cmf_artifacts/22707b00-d77c-11f1-821c-ddf24d2c...,Step_Metrics,12bd2c3831b14430234cf2bb74b71e19,Test-env:/lustre/royann/cmf/files/md5/12/bd2c3...
1,db38e93d1ec3b65588a17f2d60e2928e,1793706684573,NaN,1793706271062,NaN,9,...,RandomForestClassifier:default,RandomForestClassifier,artifacts/model/model.pkl:db38e93d1ec3b65588a1...,Model,db38e93d1ec3b65588a17f2d60e2928e,Test-env:/lustre/royann/cmf/files/md5/db/38e93...
2,NaN,1793706684615,0.572594,1793706275834,0.952194,11,...,NaN,NaN,metrics:f18760ee-d77b-11f1-8e32-1b603906cbd8:8,Metrics,f18760ee-d77b-11f1-8e32-1b603906cbd8,NaN
3,NaN,1793706684601,0.572594,1793706368368,0.952194,10,...,NaN,NaN,metrics:28af1a1c-d77c-11f1-9d1e-ed9d59c2225e:7,Metrics,28af1a1c-d77c-11f1-9d1e-ed9d59c2225e,NaN


In [19]:
# GET /artifacts/metrics/{metrics_name}
# Server-side, cmfquery.get_metrics() looks up a 'Step_Metrics' artifact by name and then reads
# its parquet file from the SERVER's filesystem (custom property 'Name'). On deployments where
# the step-metrics parquet was not pushed to the server, this correctly returns 404.
if METRICS_NAME:
    metrics = call("GET", f"/artifacts/metrics/{enc(METRICS_NAME)}")
    display(preview(metrics))
else:
    print("No Step_Metrics artifact in this pipeline — skipping.")

GET  /artifacts/metrics/cmf_artifacts%2F22707b00-d77c-11f1-821c-ddf24d2c56f8%2Fmetrics%2Ftraining_metrics%3A12bd2c3831b14430234cf2bb74b71e19%3A7%3A2578d4e6-d77c-11f1-821c-ddf24d2c56f8 -> HTTP 200 [error] Metrics not found
     errors: [{'field': 'metrics_name', 'message': "Metrics 'cmf_artifacts/22707b00-d77c-11f1-821c-ddf24d2c56f8/metrics/training_metrics:12bd2c3831b14430234cf2bb74b71e19:7:2578d4e6-d77c-11f1-821c-ddf24d2c56f8' not found", 'code': None}]


None

## Coverage summary

Every endpoint from the [CMFQuery REST API docs](https://hewlettpackard.github.io/cmf/api/public/cmfquery_rest_api/) is exercised above:

- **Pipelines (4/4):** `names`, `{p}/id`, `{p}/json` (plain and with `exec_uuid`), `sync/{t}/json`
- **Executions (11/11):** `pipeline/{p}`, `stages/name/{s}`, `stages/name/{s}/list`, `stages/id/{id}` (plain and with `execution_uuid`), `batch-get`, `batch-summary`, `id/{id}/artifacts`, `id/{id}/parents/ids`, `parents/batch-get`, `ancestors/batch-get`, `artifacts/batch-get`
- **Artifacts (15/15):** `/artifacts`, `/{p}`, `batch-get`, `name/{a}`, `name/{a}/dataframe`, `name/{a}/executions`, `id/{id}/executions`, `name/{a}/parent-executions`, `name/{a}/producer-execution`, `name/{a}/children`, `name/{a}/descendants`, `name/{a}/parents`, `id/{id}/parents`, `name/{a}/ancestors`, `metrics/{m}`

30 of 32 calls return `success`. The two expected non-successes:

- `POST /executions/ancestors/batch-get` always answers `404 Parent executions not found` — an upstream cmf bug: `cmfquery.get_all_parent_executions_by_id` iterates the result of `get_one_hop_parent_executions` expecting nested lists, but that function returns a flat list, so no ancestor is ever collected. Worth reporting on the cmf repo.
- `GET /artifacts/metrics/{metrics_name}` requires the step-metrics parquet file to exist on the **server's** filesystem (`get_metrics` does `pd.read_parquet` there). On this deployment the parquet only exists client-side, so 404 is the correct answer.